# 05 · Weight of Evidence (WoE) e Information Value (IV)
## Módulo 3 — Feature Engineering & Modelado Supervisado

---

### 🎯 Objetivo del Cuaderno
Transformar todas las variables categóricas explicativas en variables numéricas continuas mediante **Weight of Evidence (WoE)** y evaluar su poder predictivo global mediante **Information Value (IV)**:

$$\text{WoE}_i = \ln \left( \frac{\% \text{Clúster 1}_i}{\% \text{Clúster 0}_i} \right)$$

$$\text{IV} = \sum_{i=1}^k \left( \% \text{Clúster 1}_i - \% \text{Clúster 0}_i \right) \times \text{WoE}_i$$

---

### 🛡️ Propiedades Clave del WoE
1. **Tratamiento Nativo de Nulos e Incógnitas:** Los valores ausentes o de evasión (`NaN`, `No_Responde`) no se imputan a la ciega; se calculan como un grupo categórico independiente con su propio peso WoE.
2. **Relación Monótona / Lineal:** Transforma directamente las categorías de texto en *log-odds*, facilitando la estimación de parámetros en algoritmos supervisados.
3. **Imparcialidad en Test:** Si en $X_{test}$ aparece una categoría totalmente nueva e inédita en $X_{train}$, se le asigna $\text{WoE} = 0.0$ (neutralidad absoluta).

🔒 **Regla de No Contaminación (*Train-Only*):**
Los mapas de WoE y los puntajes de IV se calculan **únicamente sobre $X_{train}$**. El conjunto $X_{test}$ solo recibe el mapeo resultante mediante los diccionarios aprendidos en entrenamiento.

In [1]:
# ==============================================================================
# BLOQUE 1: CARGA DE DATOS DEPURADOS (TRAIN Y TEST DEL MÓDULO 04)
# ==============================================================================
import pandas as pd
import numpy as np

# Configuración visual
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Carga de archivos exportados en el Notebook 04
ruta_train = 'data_processed/04_train_ecommerce.csv'
ruta_test = 'data_processed/04_test_ecommerce.csv'

train_df = pd.read_csv(ruta_train)
test_df = pd.read_csv(ruta_test)

target_col = 'Cluster'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"📦 Train depurado cargado: {X_train.shape[0]} filas × {X_train.shape[1]} columnas.")
print(f"🔒 Test intacto cargado:   {X_test.shape[0]} filas × {X_test.shape[1]} columnas.")

📦 Train depurado cargado: 8018 filas × 61 columnas.
🔒 Test intacto cargado:   2022 filas × 61 columnas.


## 2. Implementación de la Función Nativa de WoE e IV

Para garantizar precisión y flexibilidad sin depender de librerías externas inestables, construimos un motor nativo que:
* Maneja categorías con frecuencia cero mediante suavizado de Laplace ($\epsilon = 0.5$).
* Devuelve tanto la **Tabla de Detalle por Categoría (WoE)** como el **Mapeo para Transformación** y el **Information Value (IV)** acumulado.

In [2]:
# ==============================================================================
# BLOQUE 2 Y 3: CÁLCULO DE WOE E IV Y RANKING PREDICTIVO OVER TRAIN
# ==============================================================================

def calcular_woe_iv_variable(X_col, y, eps=0.5):
    """
    Calcula el WoE por categoría e IV global para una sola variable categórica.
    Clase 1 = Motor Comercial, Clase 0 = Consumidor Cauto.
    """
    # Rellenamos nulos explícitamente para tratarlos como categoría
    col_serie = X_col.fillna('VALOR_FALTANTE_NA').astype(str)
    
    # Tabla de contingencia
    df_ct = pd.crosstab(col_serie, y)
    
    # Aseguramos columnas 0 y 1
    if 0 not in df_ct.columns: df_ct[0] = 0
    if 1 not in df_ct.columns: df_ct[1] = 0
        
    total_pos = (y == 1).sum()
    total_neg = (y == 0).sum()
    
    # Suavizado para evitar división por cero o log(0)
    pos_cat = df_ct[1] + eps
    neg_cat = df_ct[0] + eps
    
    tot_pos_adj = total_pos + (eps * len(df_ct))
    tot_neg_adj = total_neg + (eps * len(df_ct))
    
    dist_pos = pos_cat / tot_pos_adj
    dist_neg = neg_cat / tot_neg_adj
    
    # Cálculo de WoE
    woe = np.log(dist_pos / dist_neg)
    
    # Cálculo de IV
    iv_cat = (dist_pos - dist_neg) * woe
    iv_total = iv_cat.sum()
    
    # Mapa de reemplazo {categoria: valor_woe}
    mapa_woe = woe.to_dict()
    
    return iv_total, mapa_woe

# ------------------------------------------------------------------------------
# CÁLCULO PARA TODAS LAS VARIABLES DE X_TRAIN
# ------------------------------------------------------------------------------
dict_iv = {}
dict_mapas_woe = {}

for col in X_train.columns:
    iv_val, mapa_woe = calcular_woe_iv_variable(X_train[col], y_train)
    dict_iv[col] = iv_val
    dict_mapas_woe[col] = mapa_woe

# Construimos la tabla de resumen de IV
df_iv_summary = pd.DataFrame(list(dict_iv.items()), columns=['variable', 'IV']).sort_values(by='IV', ascending=False)

# Clasificación según estándares de la industria
def clasificar_iv(iv):
    if iv >= 0.50:
        return '🔥 Sospechoso / Extremadamente Fuerte (>0.50)'
    elif iv >= 0.30:
        return '⚡ Poder Fuerte (0.30 - 0.50)'
    elif iv >= 0.10:
        return '🔹 Poder Medio (0.10 - 0.30)'
    elif iv >= 0.02:
        return '🟢 Poder Débil / Útil (0.02 - 0.10)'
    else:
        return '🔴 Inútil (<0.02)'

df_iv_summary['Diagnostico_IV'] = df_iv_summary['IV'].apply(clasificar_iv)

print("🏆 RANKING DE INFORMATION VALUE (IV) CALCULADO SOBRE TRAIN:")
display(df_iv_summary.head(25))

print("\n" + "="*75)
print(f"· Total de variables evaluadas: {len(df_iv_summary)}")
print(f"· Variables con IV útil (≥ 0.02): {(df_iv_summary['IV'] >= 0.02).sum()}")
print(f"· Variables candidatas a descarte por IV inútil (< 0.02): {(df_iv_summary['IV'] < 0.02).sum()}")
print("="*75)

🏆 RANKING DE INFORMATION VALUE (IV) CALCULADO SOBRE TRAIN:


,variable,IV,Diagnostico_IV
0,actual_time,2.5210,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
30,primary_place_of_www_access,2.3149,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
28,primary_computing_platform,2.1464,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
24,major_occupation,2.1375,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
57,years_on_internet,1.7623,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
1,age,0.9585,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
52,web_page_creation,0.9320,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
55,who_pays_for_access_self,0.8625,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
60,who_pays_for_access_work,0.8081,🔥 Sospechoso / Extremadamente Fuerte (>0.50)
23,major_geographical_location,0.7678,🔥 Sospechoso / Extremadamente Fuerte (>0.50)



· Total de variables evaluadas: 61
· Variables con IV útil (≥ 0.02): 46
· Variables candidatas a descarte por IV inútil (< 0.02): 15


## 4. Transformación Mapeada WoE (Train y Test)

Procedemos a reemplazar las columnas categóricas originales por sus valores numéricos continuos de **WoE** usando los mapas aprendidos exclusivamente sobre $X_{train}$:

* **En $X_{train}$:** Se sustituyen las categorías por su $\text{WoE}$ correspondiente.
* **En $X_{test}$:** Se aplica el mapa de $X_{train}$. Si una categoría es totalmente nueva o no vista en entrenamiento, se le asigna $\text{WoE} = 0.0$ (neutralidad estadística).
* **Filtro de Selección Inicial:** Conservamos únicamente aquellas variables que alcanzaron un poder predictivo útil ($IV \ge 0.02$).

In [3]:
# ==============================================================================
# BLOQUE 4 Y 5: MAPEO WOE, EXPORTACIÓN DE DATASETS Y ARTEFACTOS
# ==============================================================================
import json
import joblib
import os

# 1. Filtramos la lista de variables útiles (IV >= 0.02)
vars_utiles_iv = df_iv_summary[df_iv_summary['IV'] >= 0.02]['variable'].tolist()
vars_descarte_iv = df_iv_summary[df_iv_summary['IV'] < 0.02]['variable'].tolist()

print(f"✂️ Seleccionando {len(vars_utiles_iv)} variables con IV >= 0.02 (Descartando {len(vars_descarte_iv)} variables inútiles)...")

# 2. Función de transformación WoE
def transformar_df_a_woe(df, dict_mapas, vars_a_incluir):
    df_woe = pd.DataFrame(index=df.index)
    
    for col in vars_a_incluir:
        if col in df.columns:
            mapa = dict_mapas[col]
            # Convertimos la columna a string (tratando nulos de forma explícita)
            serie_str = df[col].fillna('VALOR_FALTANTE_NA').astype(str)
            
            # Reemplazamos usando el mapa de Train; si hay categoría no vista en Test, asignamos 0.0
            df_woe[col + '_woe'] = serie_str.map(mapa).fillna(0.0).astype(float)
            
    return df_woe

# 3. Mapeamos X_train y X_test
X_train_woe = transformar_df_a_woe(X_train, dict_mapas_woe, vars_utiles_iv)
X_test_woe = transformar_df_a_woe(X_test, dict_mapas_woe, vars_utiles_iv)

# 4. Reintegramos el Target (Cluster)
train_05_woe = X_train_woe.copy()
train_05_woe[target_col] = y_train.values

test_05_woe = X_test_woe.copy()
test_05_woe[target_col] = y_test.values

# ------------------------------------------------------------------------------
# 5. GUARDADO DE BASE PROCESADA Y ARTEFACTOS DE PRODUCCIÓN
# ------------------------------------------------------------------------------
os.makedirs('data_processed', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# Guardamos los CSV procesados en formato WoE
train_05_woe.to_csv('data_processed/05_train_woe.csv', index=False)
test_05_woe.to_csv('data_processed/05_test_woe.csv', index=False)

# Guardamos el mapa de diccionarios en artifacts
joblib.dump(dict_mapas_woe, 'artifacts/woe_maps.joblib')

# Guardamos la tabla de IV en formato JSON
reporte_iv = {
    'total_variables_evaluadas': len(df_iv_summary),
    'total_variables_utiles': len(vars_utiles_iv),
    'total_variables_descartadas': len(vars_descarte_iv),
    'variables_descartadas_iv_bajo': vars_descarte_iv,
    'top_10_variables_iv': df_iv_summary.head(10).to_dict(orient='records')
}

with open('reports/iv_table.json', 'w') as f:
    json.dump(reporte_iv, f, indent=4)

print("="*75)
print("✅ DATASETS EN FORMATO WOE Y ARTEFACTOS EXPORTADOS EXITOSAMENTE:")
print(f"· Train WoE Módulo 05: {train_05_woe.shape[0]} filas × {train_05_woe.shape[1]} columnas")
print(f"· Test WoE Módulo 05:  {test_05_woe.shape[0]} filas × {test_05_woe.shape[1]} columnas")
print("· Mapa WoE guardado en: 'artifacts/woe_maps.joblib'")
print("· Reporte IV guardado en: 'reports/iv_table.json'")
print("="*75)

✂️ Seleccionando 46 variables con IV >= 0.02 (Descartando 15 variables inútiles)...
✅ DATASETS EN FORMATO WOE Y ARTEFACTOS EXPORTADOS EXITOSAMENTE:
· Train WoE Módulo 05: 8018 filas × 47 columnas
· Test WoE Módulo 05:  2022 filas × 47 columnas
· Mapa WoE guardado en: 'artifacts/woe_maps.joblib'
· Reporte IV guardado en: 'reports/iv_table.json'


## 5. Conclusiones y Diagnóstico del Information Value (IV)

La evaluación de capacidad predictiva mediante **WoE e IV** aporta las siguientes decisiones de ingeniería de datos para el pipeline:

---

### A. Capacidad Predictiva y Selección Inicial ($IV \ge 0.02$)
* **Variables Útiles Seleccionadas ($46$ variables):** El $75.4\%$ del universo de características aprobó la valla de utilidad ($IV \ge 0.02$), conservando el poder discriminante necesario para alimentar los clasificadores.
* **Poda de Variables Inútiles ($15$ variables):** Se descartaron $15$ atributos con $IV < 0.02$ que solo aportaban ruido o dimensionalidad innecesaria.
* **Explicación de $IV > 0.50$:** La presencia de variables con $IV$ extremadamente alto (`actual_time`, `primary_place_of_www_access`, `primary_computing_platform`, `years_on_internet`) no obedece a un sesgo de fuga de información (*Target Leakage*), sino a que estas dimensiones constituyen **los ejes geométricos principales sobre los cuales se construyeron los clústers en la fase no supervisada**.

---

### B. Ventajas de la Base Transformada a WoE
1. **Espacio 100% Continuo:** Todas las características cualitativas se convirtieron en variables numéricas con relación monótona directa hacia los *log-odds* del Target (`Cluster`).
2. **Resiliencia ante Ausentes:** Las categorías de omisión/nulos conservan su valor predictivo directo mapeado en su respectivo número WoE.
3. **Imparcialidad Metodológica:** Los diccionarios WoE se ajustaron **exclusivamente sobre $X_{train}$** y se aplicaron sin sesgo a $X_{test}$.

---

### 🏁 Transición al Siguiente Módulo:
Con la base evaluada, las variables inútiles descartadas y el poder predictivo (IV) documentado, se procede al cuaderno **`06_pipeline_proc.ipynb`**. 

En lugar de transformar los datos sueltos, allí construiremos el **corazón del despliegue en producción**: un **Pipeline de Preprocesamiento de Scikit-Learn** que encapsulará la imputación de nulos, el escalado y la codificación de variables, respetando la regla de oro del Machine Learning: aplicar `.fit()` **estrictamente sobre TRAIN** y `.transform()` sobre ambos conjuntos.